# SupFig4 - Visium

# Load modules 

In [ ]:
from pathlib import Path
import sys
import importlib

import anndata as an
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import squidpy as sq
from matplotlib.patches import Patch

plt.rcdefaults()

HELPER_FUNCS_DIR = Path(
    "/home/plopez/orcd/pool/Projects/Preg_placenta_VISIUM/Code_worksapce/Publication_code/helpful_functions"
)
TMA_PATH = Path(
    "/home/plopez/orcd/pool/Projects/Preg_placenta_VISIUM/data/Final_Objs/final_TMA_obj_V2.h5ad"
)
PATIENT_METADATA_PATH = Path(
    "/home/plopez/orcd/pool/Projects/Preg_placenta_VISIUM/Code_worksapce/excel_sheets/patient_metadata_V1.csv"
)
FINAL_FIG_DIR = Path(
    "/home/plopez/orcd/pool/Projects/Preg_placenta_VISIUM/results/final_figures/SupFig_3"
)

SCALE_BAR_UM = 500
SCALE_BAR_BBOX = (0.97, -0.03)
LINE_SIZE_VERTICAL = 1.2

## load functions 

In [ ]:
if str(HELPER_FUNCS_DIR) not in sys.path:
    sys.path.append(str(HELPER_FUNCS_DIR))

import vis_tcell_helpful_func as vis_tcr_plt
importlib.reload(vis_tcr_plt)

import visium_plotting as vis_plt
importlib.reload(vis_plt)   # if you edit it and want to refresh

vis_plt.configure_publication_fonts()


# load obj 

In [ ]:
TMA_obj = an.read_h5ad(TMA_PATH)


# prep metadata

In [ ]:
patient_meta = pd.read_csv(PATIENT_METADATA_PATH, header=1)


# get the crop and order of patients 

In [ ]:
## TMA2 140
tma2_140_df = {'library_id': ["TMA2-140", "TMA2-140", "TMA2-140", "TMA2-140"],
               'Patient': ['BMP-0122', 'BMP-0122', 'BMP-0064', 'BMP-0064'],
               'Region': ['Decidua', 'VC', 'Decidua', 'VC'],
               'Condition': ['PTB', "PTB", "PTB", "PTB"],
               'xmin': [280, 280, 1000, 900],

               'xmax': [780, 780, 1500, 1400],
               'ymin': [900, 1530, 875, 1530],
               'ymax': [1400, 2030, 1375, 2030]}

tma2_140_df = pd.DataFrame(tma2_140_df)
### TMA2 145
tma2_145_df = {'library_id': ["TMA2-145", "TMA2-145", "TMA2-145", "TMA2-145"],
               'Patient': ['BMP-0038', 'BMP-0038', 'BMP-0151', 'BMP-0151'],
               'Region': ['VC', 'Decidua', 'VC', 'Decidua'],
               'Condition': ['Cont_Unlbr', 'Cont_Unlbr', 'Cont_Unlbr', 'Cont_Unlbr'],
               'xmin': [350, 250, 1020, 950],
               'xmax': [850, 825, 1520, 1450],
               'ymin': [900, 1550, 975, 1550],
               'ymax': [1400, 2050, 1475, 2050]}

tma2_145_df = pd.DataFrame(tma2_145_df)
### TMA4
tma4_df = {'library_id': ['TMA4', 'TMA4', 'TMA4', 'TMA4'],
           'Patient': ['BMP-0023', 'BMP-0023', 'BMP-0077', 'BMP-0077'],
           'Region': ['VC', 'Decidua', 'VC', 'Decidua'],
           'Condition': ['Cont_Lbr', 'Cont_Lbr', 'Cont_Lbr', 'Cont_Lbr'],
           'xmin': [425, 425, 1110, 1100],
           'xmax': [925, 925, 1610, 1610],
           'ymin': [955, 1650, 975, 1610],
           'ymax': [1455, 2150, 1475, 2110]
           }

tma4_df = pd.DataFrame(tma4_df)
### TMA9
tma9_df = {'library_id': ['TMA9', 'TMA9', 'TMA9', 'TMA9'],
           'Patient': ['BMP-0121', 'BMP-0121', 'BMP-0119', 'BMP-0119'],
           'Region': ['Decidua', 'VC', 'Decidua', 'VC'],
           'Condition': ['TermPE_FGR', 'TermPE_FGR', 'PrePE_FGR', 'PrePE_FGR'],
           'xmin': [520, 510, 1150, 1150],
           'xmax': [1020, 1020, 1650, 1650],
           'ymin': [900, 1500, 950, 1525],
           'ymax': [1400, 2025, 1450, 2025]}

tma9_df = pd.DataFrame(tma9_df)

tma_df_merge = pd.concat([tma2_140_df, tma2_145_df, tma4_df, tma9_df],
                         axis=0,
                         ignore_index=True)

# Ordering patients

In [ ]:
order_patients = [
    "BMP-0023",
    "BMP-0077",
    "BMP-0038",
    "BMP-0151",
    "BMP-0119",
    "BMP-0064",
    "BMP-0122",
    "BMP-0121",
]

sample_id_map = dict(zip(patient_meta["Internal ID"], patient_meta["Sample ID"]))

plt.rcParams.update({"font.size": 15, "axes.titlesize": 15})


# Row 1 - histo only of VC

In [ ]:
n_panels = len(order_patients)
fig, axes = plt.subplots(
    1,
    n_panels,
    figsize=(24, 3.0),
    constrained_layout=False,
    gridspec_kw={"wspace": 0.02},
)
axes = np.ravel(axes)
panel_titles = []

for ax, patient in zip(axes, order_patients):
    sub_region = tma_df_merge[
        (tma_df_merge["Patient"] == patient) & (tma_df_merge["Region"] == "VC")
    ].iloc[0]

    library_id = sub_region["library_id"]
    condition = sub_region["Condition"]
    crop_coord = (
        sub_region["xmin"],
        sub_region["ymin"],
        sub_region["xmax"],
        sub_region["ymax"],
    )

    sq.pl.spatial_scatter(
        TMA_obj,
        color=None,
        img=True,
        library_key="library_id",
        library_id=library_id,
        shape="square",
        size=1,
        outline=False,
        axis_label=None,
        crop_coord=crop_coord,
        ax=ax,
        edgecolors="none",
        linewidths=0,
    )

    scale_bar_px = vis_plt.get_scale_bar_px(
        TMA_obj[TMA_obj.obs["library_id"] == library_id].copy(),
        scale_bar_um=SCALE_BAR_UM,
    )
    vis_plt.add_scale_bar(
        ax,
        scale_bar_px,
        bbox_to_anchor=SCALE_BAR_BBOX,
        bbox_transform=ax.transAxes,
        size_vertical=LINE_SIZE_VERTICAL,
        zorder=30,
    )

    ax.set_facecolor("white")
    vis_plt.clean_spatial_axis(ax, "", fontsize=16, pad=3)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color("black")
        spine.set_linewidth(1)

    leg = ax.get_legend()
    if leg is not None:
        leg.remove()

    panel_titles.append(f"{sample_id_map[patient]} - {condition}\nVC")

fig.subplots_adjust(left=0.01, right=0.995, top=0.88, bottom=0.02, wspace=0.02)

for ax, title in zip(axes, panel_titles):
    bbox = ax.get_position()
    fig.text(
        bbox.x0 + bbox.width / 2,
        bbox.y1 + 0.015,
        title,
        ha="center",
        va="bottom",
        fontsize=16,
        fontweight="bold",
    )

vis_plt.save_figure(
    fig,
    "Sup_Fig3A_top_Visium_histology_VC_row1.pdf",
    FINAL_FIG_DIR,
    format="pdf",
    dpi=600,
    bbox_inches="tight",
    pad_inches=0.01,
)
plt.close(fig)


# Row2 - dominant parent VC

In [ ]:
n_panels = len(order_patients)
fig, axes = plt.subplots(
    1,
    n_panels,
    figsize=(24, 3.0),
    constrained_layout=False,
    gridspec_kw={"wspace": 0.02},
)
axes = np.ravel(axes)

for ax, patient in zip(axes, order_patients):
    sub_region = tma_df_merge[
        (tma_df_merge["Patient"] == patient) & (tma_df_merge["Region"] == "VC")
    ].iloc[0]

    library_id = sub_region["library_id"]
    crop_coord = (
        sub_region["xmin"],
        sub_region["ymin"],
        sub_region["xmax"],
        sub_region["ymax"],
    )

    sq.pl.spatial_scatter(
        TMA_obj,
        color="dominant_parent_type",
        img=False,
        library_key="library_id",
        library_id=library_id,
        shape="square",
        size=1,
        title=None,
        outline=False,
        axis_label=None,
        crop_coord=crop_coord,
        ax=ax,
        edgecolors="none",
        linewidths=0,
    )

    scale_bar_px = vis_plt.get_scale_bar_px(
        TMA_obj[TMA_obj.obs["library_id"] == library_id].copy(),
        scale_bar_um=SCALE_BAR_UM,
    )
    vis_plt.add_scale_bar(
        ax,
        scale_bar_px,
        bbox_to_anchor=SCALE_BAR_BBOX,
        bbox_transform=ax.transAxes,
        size_vertical=LINE_SIZE_VERTICAL,
        zorder=30,
    )

    ax.set_facecolor("white")
    vis_plt.clean_spatial_axis(ax, "", fontsize=16, pad=3)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color("black")
        spine.set_linewidth(1)

    leg = ax.get_legend()
    if leg is not None:
        leg.remove()

fig.subplots_adjust(left=0.01, right=0.995, top=0.88, bottom=0.02, wspace=0.02)
vis_plt.save_figure(
    fig,
    "SupFig3A_Visium_dominant_parent_VC_row2.pdf",
    FINAL_FIG_DIR,
    format="pdf",
    dpi=600,
    bbox_inches="tight",
    pad_inches=0.01,
)
plt.close(fig)


# Row 3 - histo of Decidua

In [ ]:
n_panels = len(order_patients)
fig, axes = plt.subplots(
    1,
    n_panels,
    figsize=(24, 3.0),
    constrained_layout=False,
    gridspec_kw={"wspace": 0.02},
)
axes = np.ravel(axes)
panel_titles = []

for ax, patient in zip(axes, order_patients):
    sub_region = tma_df_merge[
        (tma_df_merge["Patient"] == patient) & (tma_df_merge["Region"] == "Decidua")
    ].iloc[0]

    library_id = sub_region["library_id"]
    condition = sub_region["Condition"]
    crop_coord = (
        sub_region["xmin"],
        sub_region["ymin"],
        sub_region["xmax"],
        sub_region["ymax"],
    )

    sq.pl.spatial_scatter(
        TMA_obj,
        color=None,
        img=True,
        library_key="library_id",
        library_id=library_id,
        shape="hex",
        size=1.5,
        outline=False,
        axis_label=None,
        crop_coord=crop_coord,
        ax=ax,
        edgecolors="none",
        linewidths=0,
    )

    scale_bar_px = vis_plt.get_scale_bar_px(
        TMA_obj[TMA_obj.obs["library_id"] == library_id].copy(),
        scale_bar_um=SCALE_BAR_UM,
    )
    vis_plt.add_scale_bar(
        ax,
        scale_bar_px,
        bbox_to_anchor=SCALE_BAR_BBOX,
        bbox_transform=ax.transAxes,
        size_vertical=LINE_SIZE_VERTICAL,
        zorder=30,
    )

    ax.set_facecolor("white")
    vis_plt.clean_spatial_axis(ax, "", fontsize=16, pad=3)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color("black")
        spine.set_linewidth(1)

    leg = ax.get_legend()
    if leg is not None:
        leg.remove()

    panel_titles.append(f"{sample_id_map[patient]} - {condition}\nVC + Decidua")

fig.subplots_adjust(left=0.01, right=0.995, top=0.88, bottom=0.02, wspace=0.02)

for ax, title in zip(axes, panel_titles):
    bbox = ax.get_position()
    fig.text(
        bbox.x0 + bbox.width / 2,
        bbox.y1 + 0.015,
        title,
        ha="center",
        va="bottom",
        fontsize=16,
        fontweight="bold",
    )

vis_plt.save_figure(
    fig,
    "Sup_Fig3B_Visium_histology_Decidua_row3.pdf",
    FINAL_FIG_DIR,
    format="pdf",
    dpi=600,
    bbox_inches="tight",
    pad_inches=0.01,
)
plt.close(fig)


# Row 4 - dominant parent decidua

In [ ]:
n_panels = len(order_patients)
fig, axes = plt.subplots(
    1,
    n_panels,
    figsize=(24, 3.0),
    constrained_layout=False,
    gridspec_kw={"wspace": 0.02},
)
axes = np.ravel(axes)

for ax, patient in zip(axes, order_patients):
    sub_region = tma_df_merge[
        (tma_df_merge["Patient"] == patient) & (tma_df_merge["Region"] == "Decidua")
    ].iloc[0]

    library_id = sub_region["library_id"]
    crop_coord = (
        sub_region["xmin"],
        sub_region["ymin"],
        sub_region["xmax"],
        sub_region["ymax"],
    )

    sq.pl.spatial_scatter(
        TMA_obj,
        color="dominant_parent_type",
        img=False,
        library_key="library_id",
        library_id=library_id,
        shape="square",
        size=1,
        title=None,
        outline=False,
        axis_label=None,
        crop_coord=crop_coord,
        ax=ax,
        edgecolors="none",
        linewidths=0,
    )

    scale_bar_px = vis_plt.get_scale_bar_px(
        TMA_obj[TMA_obj.obs["library_id"] == library_id].copy(),
        scale_bar_um=SCALE_BAR_UM,
    )
    vis_plt.add_scale_bar(
        ax,
        scale_bar_px,
        bbox_to_anchor=SCALE_BAR_BBOX,
        bbox_transform=ax.transAxes,
        size_vertical=LINE_SIZE_VERTICAL,
        zorder=30,
    )

    ax.set_facecolor("white")
    vis_plt.clean_spatial_axis(ax, "", fontsize=16, pad=3)
    for spine in ax.spines.values():
        spine.set_visible(False)

    leg = ax.get_legend()
    if leg is not None:
        leg.remove()

fig.subplots_adjust(left=0.01, right=0.995, top=0.88, bottom=0.02, wspace=0.02)
vis_plt.save_figure(
    fig,
    "Sup_Fig3B_Visium_dominant_parent_Decidua_row4.pdf",
    FINAL_FIG_DIR,
    format="pdf",
    dpi=600,
    bbox_inches="tight",
    pad_inches=0.01,
)
plt.close(fig)


# Plot legend 

In [ ]:
obs_key = "dominant_parent_type"
categories = TMA_obj.obs[obs_key].cat.categories
colors = TMA_obj.uns[f"{obs_key}_colors"]

handles = [
    Patch(facecolor=colors[i], edgecolor="none", label=cat)
    for i, cat in enumerate(categories)
]

fig, ax = plt.subplots(figsize=(8, 2))
ax.axis("off")
ax.legend(
    handles=handles,
    loc="center",
    frameon=False,
    ncol=7,
    title="Dominant parent cell type",
    title_fontsize=12,
    fontsize=10,
)

vis_plt.save_figure(
    fig,
    "SupFig3_legend_dominant_parent_cell_type.pdf",
    FINAL_FIG_DIR,
    format="pdf",
    dpi=600,
    bbox_inches="tight",
    pad_inches=0.01,
)
plt.close(fig)
